# Automatic Short Answer Grading
## Introduction
Automatic Short Answer Grading (ASAG) is a powerful deep learning tool that involves an encoder-only transformer to grade an answer compared to a received answer. For example, for the question

Question: "How can a scratch test determine which coin is harder"?
Reference answer: The harder coin will scratch the other.
Received answer: Scratch each coin with the other, and the one with fewer scratches is harder.
Response from model: 5/5

## Methodology
### Tokenization
I will use the `spaCy` library to tokenize each input from the datasets. From here, I will create a vocabulary class containing nearly all words in the datasets. This vocabulary class will also contain 3 special tokens:

- <UNK>: Unknown token
- <CLS>: Classifier token
- <SEP>: Separation token

The classifier token will be used to show the start of the input, while the separation token will separate the correct answer from the received answer and the end of the sequence.
The input will appear as follows

<CLS> [list of received answer tokens] <SEP> [list of reference answer tokens] <SEP>.

### Embedding & Encoding
Since I expect a large vocabulary size, I will use scaled embeddings so `nn.Embedding` can work without the embeddings being too similar to each other.
The model will also need to know the order of the sequence, I will use positional encoding as well.

### Multi-Headed Attention
The bulk of the model will be a multi-headed attention module comprising of numerous mult-headed attention layers. I will use <TODO: add layers> layers of it

It will go

embedding
position
decoders
maxpooling
linear (d,1)
sigmoid




In [22]:
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F

## Training Dataset -- ASAG2024

In [23]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset(
    "Meyerger/ASAG2024",
    data_files={
        "train": "train.parquet",
        "validation": "validation.parquet",
        "test": "test.parquet",
    },
)

cols = ["question", "provided_answer", "reference_answer", "data_source", "normalized_grade"]
ds = ds.select_columns(cols)
ds = ds.filter(lambda x: x["question"] is not None and x["provided_answer"] is not None)
ds = concatenate_datasets(list(ds.values()))
ds.set_format(type="torch", columns="normalized_grade")

dataLoader = DataLoader(ds, batch_size=32, shuffle=True)